# Modelo 3 — avaliar holdout (os 10 cenários, um único NER, com chunking na inferência)

Roda os **10 cenários** de uma rodada de treino (por padrão, `bertimbau-pii-v1-cenarios-com-sinteticos` — Modelo 3, 1000 reais + 500 sintéticos) contra o **holdout** (29-30 documentos nunca vistos em nenhum treino), com o mesmo critério de sempre (bbox, IoU≥0,30, regex de assinatura digital + propagação de texto idêntico) — inclusive com **chunking na inferência** (achado 2026-09-15), já que o checkpoint em si não tem chunking mas a inferência aplica de qualquer forma.

No final: tabela com F1/Precision/Recall de cada um dos 10 cenários, e o resumo de **melhor F1** e **melhor Recall** da rodada — pronto pra comparar com as outras rodadas (Modelo 4/5/6), que já foram medidas do mesmo jeito.

**Reutilizável**: para avaliar o Modelo 1 ou 2 (depois de rodados), só troque `PASTA_CENARIOS` na célula de configuração — o resto do notebook não muda.

**Pré-requisito:** venv geral do projeto (`.venv`, torch/transformers).

In [ ]:
import json
import sys
import time
from collections import defaultdict
from pathlib import Path

import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer

print(f"CUDA disponível: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Configuração

In [ ]:
def encontrar_raiz_projeto(marcador: str = "requirements.txt") -> Path:
    p = Path.cwd()
    for _ in range(6):
        if (p / marcador).exists():
            return p
        p = p.parent
    raise RuntimeError(f"Não encontrei a raiz do projeto ({marcador}) subindo a partir de {Path.cwd()}")

RAIZ = encontrar_raiz_projeto()
sys.path.insert(0, str(RAIZ / "scripts"))
from pos_processamento_ner import (
    extrair_nome_assinatura_digital,
    propagar_entidades_por_texto_identico,
    mesclar_previsoes_por_janela,
)

# --- Aponte para a pasta de cenários que quer avaliar -- troque aqui pra reaproveitar este
# notebook pro Modelo 1 ou Modelo 2 (depois de rodados) ---
PASTA_CENARIOS = RAIZ / "models" / "bertimbau-pii-v1-cenarios-com-sinteticos"  # Modelo 3 (1000 reais + 500 sintéticos)
# PASTA_CENARIOS = RAIZ / "models" / "bertimbau-pii-v1-modelo1-rerun-cenarios"  # Modelo 1 rerun
# PASTA_CENARIOS = RAIZ / "models" / "bertimbau-pii-v1-modelo2-rerun-cenarios"  # Modelo 2 rerun

MODELO_GABARITO = "gemini-2.5-flash"
TIPOS = ["CPF", "RG", "NOME", "ENDERECO"]
IOU_MINIMO_MATCH = 0.30
MAX_LENGTH = 512
LIMIAR_ALINHAMENTO_MINIMO_PCT = 80

PASTA_EXTRACAO_HOLDOUT = RAIZ / "data" / "documento_social" / "teste_holdout" / "extracao"
PASTA_GABARITO_HOLDOUT = RAIZ / "data" / "documento_social" / "teste_holdout" / "gabarito_alinhado" / MODELO_GABARITO

ARQUIVO_SAIDA = PASTA_CENARIOS / "_avaliacao_holdout.json"

print(f"Avaliando cenários em: {PASTA_CENARIOS}")
print(f"Existe? {PASTA_CENARIOS.exists()}")


## Carrega o gabarito alinhado + extração do holdout (idêntico aos notebooks 10/11)

In [ ]:
def bbox_contem_centro(caixa: list, bbox_token: list) -> bool:
    x0, y0, x1, y1 = caixa
    cx = (bbox_token[0] + bbox_token[2]) / 2
    cy = (bbox_token[1] + bbox_token[3]) / 2
    return (x0 - 1) <= cx <= (x1 + 1) and (y0 - 1) <= cy <= (y1 + 1)


def construir_exemplos_documento(extracao: dict, gabarito: dict) -> list:
    entidades_por_pagina = {}
    for e in gabarito["entities"]:
        if e.get("match_type") == "unmatched":
            continue
        entidades_por_pagina.setdefault(e["page"], []).append(e)
    exemplos = []
    for pagina_dados in extracao["pages"]:
        tokens = pagina_dados["tokens"]
        if not tokens:
            continue
        exemplos.append({
            "document_id": extracao["document_id"],
            "page": pagina_dados["page"],
            "tokens": [t["text"] for t in tokens],
            "tokens_bbox": [t["bbox"] for t in tokens],
        })
    return exemplos


arquivos_gabarito = sorted(f for f in PASTA_GABARITO_HOLDOUT.glob("*.json") if f.name != "_nao_alinhadas.json")
if not arquivos_gabarito:
    raise SystemExit(f"Nenhum gabarito alinhado encontrado em {PASTA_GABARITO_HOLDOUT} -- rode os notebooks 07 e 09 (em notebooks/bert/) antes.")

exemplos_holdout = []
gabarito_por_doc_pagina = {}
for arq_gab in arquivos_gabarito:
    arq_ext = PASTA_EXTRACAO_HOLDOUT / arq_gab.name
    if not arq_ext.exists():
        print(f"[AVISO] Sem extração local correspondente para {arq_gab.name}, pulando.")
        continue
    gabarito = json.loads(arq_gab.read_text(encoding="utf-8"))
    entidades = gabarito.get("entities", [])
    n_unmatched = sum(1 for e in entidades if e.get("match_type") == "unmatched")
    pct_alinhado = 100 * (len(entidades) - n_unmatched) / len(entidades) if entidades else 100
    if entidades and pct_alinhado < LIMIAR_ALINHAMENTO_MINIMO_PCT:
        print(f"[AVISO] {arq_gab.name}: só {pct_alinhado:.0f}% alinhado, abaixo do limiar -- pulando.")
        continue
    extracao = json.loads(arq_ext.read_text(encoding="utf-8"))
    exemplos_holdout.extend(construir_exemplos_documento(extracao, gabarito))
    for e in entidades:
        if e.get("match_type") == "unmatched":
            continue
        gabarito_por_doc_pagina.setdefault((gabarito["document_id"], e["page"]), []).append(e)

print(f"Documentos de holdout com gabarito utilizável: {len({ex['document_id'] for ex in exemplos_holdout})} | páginas: {len(exemplos_holdout)}")


## Funções de inferência (com chunking), decodificação BIO→bbox e métricas

In [ ]:
def agrupar_bboxes_por_linha(tokens_bbox: list, indices: list) -> list:
    if not indices:
        return []
    grupos = []
    grupo_atual = [tokens_bbox[indices[0]]]
    y_ref = tokens_bbox[indices[0]][1]
    for i in indices[1:]:
        bbox = tokens_bbox[i]
        altura_ref = grupo_atual[-1][3] - grupo_atual[-1][1]
        if abs(bbox[1] - y_ref) > max(altura_ref * 0.6, 3):
            grupos.append(grupo_atual)
            grupo_atual = [bbox]
            y_ref = bbox[1]
        else:
            grupo_atual.append(bbox)
    grupos.append(grupo_atual)
    return [[min(b[0] for b in g), min(b[1] for b in g), max(b[2] for b in g), max(b[3] for b in g)] for g in grupos]


def decodificar_bio(tags_previstas: list, tokens_bbox: list, tokens: list) -> list:
    entidades = []
    i, n = 0, len(tags_previstas)
    while i < n:
        tag = tags_previstas[i]
        if tag == "O":
            i += 1
            continue
        tipo = tag.split("-", 1)[1]
        indices = [i]
        j = i + 1
        while j < n and tags_previstas[j] == f"I-{tipo}":
            indices.append(j)
            j += 1
        entidades.append({
            "type": tipo,
            "bbox": agrupar_bboxes_por_linha(tokens_bbox, indices),
            "indices": indices,
            "text": " ".join(tokens[k] for k in indices),
        })
        i = j
    return entidades


def bbox_externo(lista_bboxes: list):
    if not lista_bboxes:
        return None
    return [min(b[0] for b in lista_bboxes), min(b[1] for b in lista_bboxes),
            max(b[2] for b in lista_bboxes), max(b[3] for b in lista_bboxes)]


def iou(a, b) -> float:
    if a is None or b is None:
        return 0.0
    xi0, yi0 = max(a[0], b[0]), max(a[1], b[1])
    xi1, yi1 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, xi1 - xi0) * max(0, yi1 - yi0)
    if inter == 0:
        return 0.0
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    uniao = area_a + area_b - inter
    return inter / uniao if uniao > 0 else 0.0


def avaliar_com_pos_processamento(predicoes_por_pagina: dict, exemplos: list) -> dict:
    """Base + regex de assinatura digital + propagação de texto idêntico -- mesmo pipeline
    completo dos notebooks 10/11 (sempre com as duas correções, que é o que os modelos 4/5/6
    também usam)."""
    contadores = {t: {"tp": 0, "fp": 0, "fn": 0} for t in TIPOS}
    exemplos_por_doc = defaultdict(list)
    for ex in exemplos:
        exemplos_por_doc[ex["document_id"]].append(ex)

    for doc_id, exs in exemplos_por_doc.items():
        previstas_por_pagina = {}
        entidades_documento = []
        for ex in exs:
            previstas = [dict(p) for p in predicoes_por_pagina[(doc_id, ex["page"])]]
            extras = extrair_nome_assinatura_digital(ex["tokens"], ex["tokens_bbox"])
            for extra in extras:
                bbox_extra = bbox_externo(extra["bbox"])
                ja_coberto = any(
                    p["type"] == "NOME" and iou(bbox_extra, bbox_externo(p["bbox"])) > 0.1
                    for p in previstas
                )
                if not ja_coberto:
                    previstas.append(extra)
            for p in previstas:
                p["page"] = ex["page"]
            previstas_por_pagina[ex["page"]] = previstas
            entidades_documento.extend(previstas)

        paginas_doc = [
            {"page": ex["page"], "tokens": [{"text": t, "bbox": b} for t, b in zip(ex["tokens"], ex["tokens_bbox"])]}
            for ex in exs
        ]
        entidades_documento = propagar_entidades_por_texto_identico(
            entidades_documento, paginas_doc, tipos_propagaveis=tuple(TIPOS)
        )
        previstas_por_pagina = defaultdict(list)
        for p in entidades_documento:
            previstas_por_pagina[p["page"]].append(p)

        for ex in exs:
            previstas = previstas_por_pagina.get(ex["page"], [])
            gold = gabarito_por_doc_pagina.get((doc_id, ex["page"]), [])
            previstas_por_tipo = {}
            for p in previstas:
                previstas_por_tipo.setdefault(p["type"], []).append(p)

            usadas = set()
            for g in gold:
                tipo = g["type"]
                candidatos = previstas_por_tipo.get(tipo, [])
                bbox_gold = bbox_externo(g.get("bbox") or [])
                melhor_iou, melhor_idx = 0.0, None
                for i, p in enumerate(candidatos):
                    if id(p) in usadas:
                        continue
                    score = iou(bbox_gold, bbox_externo(p["bbox"]))
                    if score > melhor_iou:
                        melhor_iou, melhor_idx = score, i
                if melhor_idx is not None and melhor_iou >= IOU_MINIMO_MATCH:
                    usadas.add(id(candidatos[melhor_idx]))
                    contadores[tipo]["tp"] += 1
                else:
                    contadores[tipo]["fn"] += 1

            for tipo, candidatos in previstas_por_tipo.items():
                for p in candidatos:
                    if id(p) not in usadas:
                        contadores.setdefault(tipo, {"tp": 0, "fp": 0, "fn": 0})
                        contadores[tipo]["fp"] += 1
    return contadores


def metricas_geral(contadores: dict) -> dict:
    tp_g = sum(c["tp"] for c in contadores.values())
    fp_g = sum(c["fp"] for c in contadores.values())
    fn_g = sum(c["fn"] for c in contadores.values())
    precision = tp_g / (tp_g + fp_g) if (tp_g + fp_g) else 0.0
    recall = tp_g / (tp_g + fn_g) if (tp_g + fn_g) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"tp": tp_g, "fp": fp_g, "fn": fn_g, "precision": precision, "recall": recall, "f1": f1}


def metricas_por_tipo(contadores: dict) -> dict:
    resultado = {}
    for tipo in TIPOS:
        c = contadores.get(tipo, {"tp": 0, "fp": 0, "fn": 0})
        tp, fp, fn = c["tp"], c["fp"], c["fn"]
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        resultado[tipo] = {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1}
    return resultado


## Avalia um cenário (carrega o modelo, roda inferência com chunking, aplica pós-processamento)

In [ ]:
def avaliar_cenario(nome_cenario: str) -> dict:
    pasta_modelo = PASTA_CENARIOS / nome_cenario
    print(f"\n=== Avaliando {nome_cenario} ===")
    inicio = time.time()

    tokenizer = AutoTokenizer.from_pretrained(pasta_modelo)
    model = AutoModelForTokenClassification.from_pretrained(pasta_modelo).to(device)
    model.eval()

    label_list = list(model.config.id2label.values())
    id2label = {i: l for i, l in enumerate(label_list)}

    @torch.no_grad()
    def _prever_tags_janela(tokens_janela: list) -> list:
        encoding = tokenizer(tokens_janela, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
        encoding = {k: v.to(device) for k, v in encoding.items() if k in ("input_ids", "attention_mask", "token_type_ids")}
        logits = model(**encoding).logits[0]
        pred_ids = logits.argmax(dim=-1).tolist()
        word_ids = tokenizer(tokens_janela, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH).word_ids()
        tags = ["O"] * len(tokens_janela)
        visto = set()
        for pid, wid in zip(pred_ids, word_ids):
            if wid is None or wid in visto:
                continue
            visto.add(wid)
            tags[wid] = id2label[pid]
        return tags

    def prever_pagina(tokens: list, tokens_bbox: list) -> list:
        if not tokens:
            return []
        tags = mesclar_previsoes_por_janela(tokens, tokenizer, _prever_tags_janela, max_length=MAX_LENGTH)
        return decodificar_bio(tags, tokens_bbox, tokens)

    predicoes_por_pagina = {}
    for ex in exemplos_holdout:
        predicoes_por_pagina[(ex["document_id"], ex["page"])] = prever_pagina(ex["tokens"], ex["tokens_bbox"])

    contadores = avaliar_com_pos_processamento(predicoes_por_pagina, exemplos_holdout)
    geral = metricas_geral(contadores)
    por_tipo = metricas_por_tipo(contadores)

    duracao = time.time() - inicio
    print(f"{nome_cenario}: F1={geral['f1']:.4f}  P={geral['precision']:.4f}  R={geral['recall']:.4f}  ({duracao:.0f}s)")
    for tipo in TIPOS:
        m = por_tipo[tipo]
        print(f"    {tipo:10s} F1={m['f1']:.3f}  P={m['precision']:.3f}  R={m['recall']:.3f}")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {"geral": geral, "por_tipo": por_tipo}


## Roda os 10 cenários e monta a tabela final

In [ ]:
nomes_cenarios = sorted(p.name for p in PASTA_CENARIOS.iterdir() if p.is_dir() and (p / "config.json").exists())
print(f"{len(nomes_cenarios)} cenário(s) encontrado(s) em {PASTA_CENARIOS}: {nomes_cenarios}")

resultados = {}
if ARQUIVO_SAIDA.exists():
    resultados = json.loads(ARQUIVO_SAIDA.read_text(encoding="utf-8"))
    print(f"Reaproveitando {len(resultados)} cenário(s) já avaliado(s) anteriormente.")

for nome in nomes_cenarios:
    if nome in resultados:
        print(f"{nome} já avaliado, pulando.")
        continue
    resultados[nome] = avaliar_cenario(nome)
    ARQUIVO_SAIDA.write_text(json.dumps(resultados, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"\nResultados salvos em: {ARQUIVO_SAIDA}")


## Resumo final: tabela dos 10 cenários + melhor F1 / melhor Recall da rodada

In [ ]:
print(f"{'Cenário':<36}{'F1':<8}{'Precision':<12}{'Recall':<8}")
print("-" * 64)
for nome, r in resultados.items():
    g = r["geral"]
    print(f"{nome:<36}{g['f1']:<8.4f}{g['precision']:<12.4f}{g['recall']:<8.4f}")

melhor_f1 = max(resultados.items(), key=lambda kv: kv[1]["geral"]["f1"])
melhor_recall = max(resultados.items(), key=lambda kv: kv[1]["geral"]["recall"])

print(f"\nMelhor F1:     {melhor_f1[0]} -- F1={melhor_f1[1]['geral']['f1']:.4f}  Recall={melhor_f1[1]['geral']['recall']:.4f}  Precision={melhor_f1[1]['geral']['precision']:.4f}")
print(f"Melhor Recall: {melhor_recall[0]} -- Recall={melhor_recall[1]['geral']['recall']:.4f}  F1={melhor_recall[1]['geral']['f1']:.4f}  Precision={melhor_recall[1]['geral']['precision']:.4f}")
